# Data Acquisition and Cleaning

Author: Molly

## Purpose
This notebook documents data acquisition, quality checks,
and cleaning for the Boston Streetscape × Traffic Crashes project.

## Planned steps
1. Load the raw data.
2. Inspect columns, data types, and missing values.
3. Check duplicates and invalid values.
4. Apply and document cleaning decisions.
5. Save the cleaned data to data/processed/.

In [3]:
import requests

road_url = (
    "https://gis.massdot.state.ma.us/arcgis/rest/services/"
    "Roads/RoadInventoryYearEndFiles/FeatureServer/8"
)

response = requests.get(road_url, params={"f": "json"}, timeout=60)
response.raise_for_status()
metadata = response.json()

if "error" in metadata:
    print("API error:", metadata["error"])
else:
    print("Layer name:", metadata["name"])
    print("Geometry type:", metadata["geometryType"])
    print("Number of fields:", len(metadata["fields"]))

Layer name: Road Inventory 2022
Geometry type: esriGeometryPolyline
Number of fields: 75


In [4]:
fields_df = pd.DataFrame([
    {
        "field_name": field["name"],
        "description": field.get("alias"),
        "data_type": field["type"]
    }
    for field in metadata["fields"]
])

display(fields_df)

,field_name,description,data_type
0,route_id,Route ID,esriFieldTypeString
1,from_measure,From Measure,esriFieldTypeDouble
2,to_measure,To Measure,esriFieldTypeDouble
3,Route_System,Route_System,esriFieldTypeString
4,Route_Number,Route_Number,esriFieldTypeString
...,...,...,...
70,Local_Priority,Local_Priority,esriFieldTypeSmallInteger
71,Oneway,Oneway,esriFieldTypeString
72,Structure,Structure Type,esriFieldTypeSmallInteger
73,OBJECTID,OBJECTID,esriFieldTypeOID


In [5]:
with pd.option_context("display.max_rows", None):
    display(fields_df)

,field_name,description,data_type
0,route_id,Route ID,esriFieldTypeString
1,from_measure,From Measure,esriFieldTypeDouble
2,to_measure,To Measure,esriFieldTypeDouble
3,Route_System,Route_System,esriFieldTypeString
4,Route_Number,Route_Number,esriFieldTypeString
5,Route_Direction,Route_Direction,esriFieldTypeString
6,Rd_Seg_ID,Road Segment ID,esriFieldTypeInteger
7,Facility,Facility Type,esriFieldTypeSmallInteger
8,Mile_Count,Mileage Counted,esriFieldTypeSmallInteger
9,F_Class,Functional Class,esriFieldTypeSmallInteger


In [6]:
params = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": "true",
    "orderByFields": "OBJECTID ASC",
    "resultRecordCount": 5,
    "f": "json"
}

response = requests.get(
    road_url + "/query",
    params=params,
    timeout=60
)
response.raise_for_status()
sample_data = response.json()

if "error" in sample_data:
    print("API error:", sample_data["error"])
else:
    sample_df = pd.DataFrame(
        feature["attributes"]
        for feature in sample_data["features"]
    )

    print("取得筆數:", len(sample_df))
    display(sample_df)

取得筆數: 5


,route_id,from_measure,to_measure,Route_System,Route_Number,Route_Direction,Rd_Seg_ID,Facility,Mile_Count,F_Class,...,created_date,last_edited_user,last_edited_date,Record_ID,GlobalID,Local_Priority,Oneway,Structure,OBJECTID,Shape__Length
0,B0001,0.000000,1.719919,B,0001,None,None,12,0,0,...,1677824580000,GEODOTADMIN,1677824580000,60440.917972.NE.27.0001,{15A16E80-A960-479B-8C51-D0E92786B9A9},1,None,None,1,2767.946996
1,B0001,1.719919,2.102563,B,0001,None,None,12,0,0,...,1677824580000,GEODOTADMIN,1677824580000,60861.919612.NE.61.0001,{6670CBAD-C5DC-4FC2-9211-C9C7AA436E5F},1,None,None,2,615.806771
2,B0001,2.102563,2.378992,B,0001,None,None,12,0,0,...,1677824580000,GEODOTADMIN,1677824580000,60874.920142.NE.44.0001,{9A789E46-705F-4EAF-AEF0-DB0439E355A5},1,None,None,3,444.870559
3,B0001,2.378992,3.676241,B,0001,None,None,12,0,0,...,1677824580000,GEODOTADMIN,1677824580000,61188.921378.NE.20.0001,{2F040FDC-708D-4826-AC61-33DA9786BAB0},1,None,None,4,2087.723529
4,B0001,3.676241,4.082555,B,0001,None,None,12,0,0,...,1677824580000,GEODOTADMIN,1677824580000,62090.922342.NE.65.0001,{7C01F298-6B78-412F-8F47-74C091BE4433},1,None,None,5,653.900662


In [7]:
columns_to_view = [
    "OBJECTID", "Record_ID", "St_Name", "City",
    "Facility", "F_Class", "Num_Lanes", "Surface_Wd",
    "Speed_Lim", "Speed", "AADT", "AADT_Year",
    "Length", "Shape__Length"
]

display(sample_df[columns_to_view])

,OBJECTID,Record_ID,St_Name,City,Facility,F_Class,Num_Lanes,Surface_Wd,Speed_Lim,Speed,AADT,AADT_Year,Length,Shape__Length
0,1,60440.917972.NE.27.0001,,148,12,0,0,0.0,None,None,None,None,1.719919,2767.946996
1,2,60861.919612.NE.61.0001,,148,12,0,0,0.0,None,None,None,None,0.382644,615.806771
2,3,60874.920142.NE.44.0001,,58,12,0,0,0.0,None,None,None,None,0.276429,444.870559
3,4,61188.921378.NE.20.0001,,58,12,0,0,0.0,None,None,None,None,1.297249,2087.723529
4,5,62090.922342.NE.65.0001,,58,12,0,0,0.0,None,None,None,None,0.406314,653.900662


In [8]:
facility_field = next(
    field for field in metadata["fields"]
    if field["name"] == "Facility"
)

facility_domain = facility_field.get("domain") or {}
facility_codes = facility_domain.get("codedValues", [])

display(pd.DataFrame(facility_codes))

,name,code
0,Mainline roadway,1
1,Tunnel,3
2,Doubledeck,4
3,Rotary,5
4,Causeway,6
5,Collector - Distributor,10
6,Ramp - NB/EB,8
7,Ramp - SB/WB,9
8,Simple Ramp - Tunnel,11
9,Roundabout,2


In [9]:
road_params = params.copy()
road_params["where"] = "Facility = 1"

response = requests.get(
    road_url + "/query",
    params=road_params,
    timeout=60
)
response.raise_for_status()
road_sample = response.json()

if "error" in road_sample:
    print("API error:", road_sample["error"])
else:
    road_sample_df = pd.DataFrame(
        feature["attributes"]
        for feature in road_sample["features"]
    )
    display(road_sample_df[columns_to_view])

,OBJECTID,Record_ID,St_Name,City,Facility,F_Class,Num_Lanes,Surface_Wd,Speed_Lim,Speed,AADT,AADT_Year,Length,Shape__Length
0,3418,238110.910926.NE.22.1515,PENNEY HILL ROAD,178,1,0,1,14.0,NaN,None,1154,NaN,0.133844,221.951441
1,3998,175704.893321.NW.89.190-DUP,INTERSTATE 190,348,1,1,2,36.0,35.0,None,8030,2021.0,0.553178,890.253746
2,3999,175433.893155.SE.44.190-DUP,INTERSTATE 190,348,1,1,2,36.0,35.0,None,73122,2014.0,0.275647,443.610427
3,4000,175820.893289.NE.45.190-DUP,INTERSTATE 190,348,1,1,2,36.0,35.0,None,73122,2014.0,0.283969,457.003551
4,4001,175455.892712.NE.31.190,INTERSTATE 190,348,1,1,2,24.0,50.0,None,72037,2021.0,0.197255,317.450485


## Initial observations from test records

- The first five unfiltered records were bicycle/pedestrian facilities
  (Facility = 12).
- Five additional mainline roadway records (Facility = 1) were inspected.
- Missing values were observed in speed-limit and AADT measurement-year fields.
- AADT measurement years vary within the 2022 inventory snapshot.
- Some Record_ID values contain "DUP"; their meaning needs verification
  before any deduplication.

These are small test samples, not a representative study dataset.
No records have been removed or imputed.

In [10]:
city_field = next(
    field for field in metadata["fields"]
    if field["name"] == "City"
)

city_domain = city_field.get("domain") or {}
city_codes_df = pd.DataFrame(city_domain.get("codedValues", []))

display(
    city_codes_df[
        city_codes_df["name"].str.contains(
            "Boston", case=False, na=False
        )
    ]
)

,name,code
34,Boston,35


In [11]:
boston_where = "City = 35"

response = requests.get(
    road_url + "/query",
    params={
        "where": boston_where,
        "returnCountOnly": "true",
        "f": "json"
    },
    timeout=60
)
response.raise_for_status()
count_result = response.json()

if "error" in count_result:
    print("API error:", count_result["error"])
else:
    print("Boston 道路紀錄總筆數:", count_result["count"])

Boston 道路紀錄總筆數: 22627


In [12]:
boston_features = []
batch_size = 1000
expected_count = count_result["count"]

for offset in range(0, expected_count, batch_size):
    response = requests.get(
        road_url + "/query",
        params={
            "where": boston_where,
            "outFields": "*",
            "returnGeometry": "true",
            "orderByFields": "OBJECTID ASC",
            "resultOffset": offset,
            "resultRecordCount": batch_size,
            "f": "json"
        },
        timeout=60
    )
    response.raise_for_status()
    batch = response.json()

    if "error" in batch:
        raise RuntimeError(batch["error"])

    features = batch.get("features", [])
    if not features:
        raise RuntimeError("提前收到空資料，請檢查下載是否完整。")

    boston_features.extend(features)
    print(f"已取得 {len(boston_features):,} / {expected_count:,} 筆")

object_ids = [
    feature["attributes"]["OBJECTID"]
    for feature in boston_features
]

assert len(boston_features) == expected_count, "下載筆數不符"
assert len(set(object_ids)) == expected_count, "出現重複 OBJECTID"

print("完成：筆數正確，OBJECTID 沒有重複。")

已取得 1,000 / 22,627 筆
已取得 2,000 / 22,627 筆
已取得 3,000 / 22,627 筆
已取得 4,000 / 22,627 筆
已取得 5,000 / 22,627 筆
已取得 6,000 / 22,627 筆
已取得 7,000 / 22,627 筆
已取得 8,000 / 22,627 筆
已取得 9,000 / 22,627 筆
已取得 10,000 / 22,627 筆
已取得 11,000 / 22,627 筆
已取得 12,000 / 22,627 筆
已取得 13,000 / 22,627 筆
已取得 14,000 / 22,627 筆
已取得 15,000 / 22,627 筆
已取得 16,000 / 22,627 筆
已取得 17,000 / 22,627 筆
已取得 18,000 / 22,627 筆
已取得 19,000 / 22,627 筆
已取得 20,000 / 22,627 筆
已取得 21,000 / 22,627 筆
已取得 22,000 / 22,627 筆
已取得 22,627 / 22,627 筆
完成：筆數正確，OBJECTID 沒有重複。


In [13]:
import json
from pathlib import Path
from datetime import datetime, timezone

# 從專案根目錄或 notebooks 內執行，都能找到專案
cwd = Path.cwd()
project_root = next(
    (p for p in [cwd, *cwd.parents] if (p / ".git").exists()),
    None
)
if project_root is None:
    raise RuntimeError("找不到專案根目錄，請確認開啟的是專案資料夾。")

raw_dir = project_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

download_time = datetime.now(timezone.utc)
stamp = download_time.strftime("%Y%m%dT%H%M%S%fZ")
output_path = raw_dir / f"boston_roads_2022_raw_{stamp}.json"

raw_data = {
    "source_url": road_url,
    "filter": boston_where,
    "downloaded_at_utc": download_time.isoformat(),
    "record_count": len(boston_features),
    "layer_metadata": metadata,
    "features": boston_features
}

with output_path.open("x", encoding="utf-8") as file:
    json.dump(raw_data, file, ensure_ascii=False)

print("原始資料已儲存至:", output_path)

原始資料已儲存至: c:\Users\wendy\Desktop\Boston University\classes\CS506\Final_Project\cs506-final-project\data\raw\boston_roads_2022_raw_20261016T170046534733Z.json


In [14]:
# 把道路屬性整理成表格；座標仍保留在原始資料中
boston_df = pd.DataFrame(
    feature["attributes"]
    for feature in boston_features
)

facility_names = {
    item["code"]: item["name"]
    for item in facility_codes
}

facility_summary = (
    boston_df.groupby("Facility", dropna=False)
    .size()
    .reset_index(name="record_count")
)

facility_summary["facility_name"] = (
    facility_summary["Facility"].map(facility_names)
)

display(
    facility_summary.sort_values("record_count", ascending=False)
)

,Facility,record_count,facility_name
0,1.0,18398,Mainline roadway
10,NaN,3288,NaN
4,7.0,390,Simple Ramp/ Channelized Turning Lane
9,12.0,246,Bicycle/ Pedestrian
1,3.0,96,Tunnel
3,5.0,74,Rotary
8,11.0,73,Simple Ramp - Tunnel
5,8.0,25,Ramp - NB/EB
2,4.0,21,Doubledeck
7,10.0,11,Collector - Distributor


In [15]:
check_columns = [
    "Record_ID", "St_Name", "City", "Facility",
    "F_Class", "Num_Lanes", "Surface_Wd",
    "Speed_Lim", "Speed", "AADT", "AADT_Year", "Length"
]

# 建立檢查用副本，把空字串或只有空白的內容也算成缺失
check_df = boston_df[check_columns].copy()
check_df = check_df.replace(r"^\s*$", pd.NA, regex=True)

missing_summary = pd.DataFrame({
    "missing_count": check_df.isna().sum(),
    "missing_percent": (
        check_df.isna().mean() * 100
    ).round(2)
})

display(
    missing_summary.sort_values("missing_count", ascending=False)
)

,missing_count,missing_percent
Speed,22363,98.83
AADT_Year,14104,62.33
Speed_Lim,13597,60.09
AADT,4206,18.59
Facility,3288,14.53
St_Name,2800,12.37
Surface_Wd,657,2.90
Num_Lanes,472,2.09
F_Class,410,1.81
Record_ID,37,0.16


In [16]:
has_aadt = check_df["AADT"].notna()
has_year = check_df["AADT_Year"].notna()

aadt_summary = pd.DataFrame({
    "status": [
        "AADT and year available",
        "AADT available, year missing",
        "AADT missing, year available",
        "Both missing"
    ],
    "record_count": [
        (has_aadt & has_year).sum(),
        (has_aadt & ~has_year).sum(),
        (~has_aadt & has_year).sum(),
        (~has_aadt & ~has_year).sum()
    ]
})

display(aadt_summary)

,status,record_count
0,AADT and year available,8522
1,"AADT available, year missing",9899
2,"AADT missing, year available",1
3,Both missing,4205


In [ ]:
aadt_years = pd.to_numeric(
    check_df["AADT_Year"], errors="coerce"
)

display(
    aadt_years.value_counts()
    .sort_index()
    .rename_axis("AADT_Year")
    .reset_index(name="record_count")
)

print("年份晚於 2022 的筆數:", (aadt_years > 2022).sum())

,AADT_Year,record_count
0,2009.0,480
1,2011.0,8
2,2012.0,2
3,2013.0,11
4,2014.0,155
5,2015.0,3
6,2016.0,37
7,2017.0,43
8,2019.0,8
9,2020.0,124


年份晚於 2022 的筆數: 0


## Boston road inventory: initial data-quality findings

- Downloaded 22,627 Boston records from the Road Inventory 2022 layer.
- The downloaded count matched the API count, with no duplicate OBJECTIDs.
- Facility type is missing in 3,288 records (14.53%).
- Street name is missing or blank in 2,800 records (12.37%).
- AADT is missing in 4,206 records (18.59%).
- 9,899 records have AADT but lack its measurement year.
- Reported AADT years range from 2009 to 2021, with most dated 2021.
- No reported AADT year exceeds 2022.
- Speed_Lim is missing in 60.09% of records; Speed is missing in 98.83%.

These findings cover all downloaded Boston records, before eligibility filtering.
Missing-value codes and field definitions still require verification.
No records have been removed, and no missing values have been imputed.

In [19]:
id_summary = []

for column in ["OBJECTID", "Record_ID", "Rd_Seg_ID", "GlobalID"]:
    values = boston_df[column].replace(r"^\s*$", pd.NA, regex=True)
    available = values.dropna()

    id_summary.append({
        "field": column,
        "missing_count": values.isna().sum(),
        "unique_non_missing": available.nunique(),
        "rows_with_repeated_id": available.duplicated(keep=False).sum()
    })

display(pd.DataFrame(id_summary))

,field,missing_count,unique_non_missing,rows_with_repeated_id
0,OBJECTID,0,22627,0
1,Record_ID,37,22590,0
2,Rd_Seg_ID,13699,6660,3925
3,GlobalID,0,22627,0


In [20]:
roads_work = boston_df.copy()

roads_work["segment_id"] = roads_work["GlobalID"]

assert roads_work["segment_id"].notna().all()
assert roads_work["segment_id"].is_unique

display(
    roads_work[
        ["segment_id", "OBJECTID", "Record_ID", "Rd_Seg_ID"]
    ].head()
)

,segment_id,OBJECTID,Record_ID,Rd_Seg_ID
0,{DDF2F037-CFE4-4221-9734-1208F20D34FB},843,235207.904232.NW.12.0240,NaN
1,{D4A74835-5611-4F24-9CFB-EEE9B2E3BF4E},845,235191.904261.NW.36.0240,NaN
2,{44C09551-3E8C-44C4-8F91-D289F2E3B929},847,235126.904410.NW.30.0240,NaN
3,{AFF012D1-B4B0-4F58-8421-B0A4E812109E},866,241784.904754.SE.22.0246,NaN
4,{26B1BB5F-BDE3-4370-B298-081371A00959},867,241686.904691.SW.13.0247,NaN


In [21]:
# 保留原始道路名稱，方便回頭比對
roads_work["street_name_raw"] = roads_work["St_Name"]

# 移除名稱前後空白，將空字串統一為缺失值
roads_work["street_name_clean"] = (
    roads_work["St_Name"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

print(
    "缺少道路名稱的筆數:",
    roads_work["street_name_clean"].isna().sum()
)

display(
    roads_work[
        ["segment_id", "street_name_raw", "street_name_clean"]
    ].head()
)

缺少道路名稱的筆數: 2800


,segment_id,street_name_raw,street_name_clean
0,{DDF2F037-CFE4-4221-9734-1208F20D34FB},,<NA>
1,{D4A74835-5611-4F24-9CFB-EEE9B2E3BF4E},,<NA>
2,{44C09551-3E8C-44C4-8F91-D289F2E3B929},,<NA>
3,{AFF012D1-B4B0-4F58-8421-B0A4E812109E},,<NA>
4,{26B1BB5F-BDE3-4370-B298-081371A00959},,<NA>


In [22]:
roads_work["street_name_missing"] = (
    roads_work["street_name_clean"].isna()
)
roads_work["aadt_missing"] = roads_work["AADT"].isna()
roads_work["aadt_year_missing"] = roads_work["AADT_Year"].isna()
roads_work["facility_missing"] = roads_work["Facility"].isna()

flag_columns = [
    "street_name_missing",
    "aadt_missing",
    "aadt_year_missing",
    "facility_missing"
]

display(
    roads_work[flag_columns]
    .sum()
    .rename("flagged_count")
    .to_frame()
)

,flagged_count
street_name_missing,2800
aadt_missing,4206
aadt_year_missing,14104
facility_missing,3288


In [23]:
numeric_columns = [
    "Length", "Shape__Length", "Num_Lanes",
    "Surface_Wd", "Speed_Lim", "Speed", "AADT"
]

numeric_checks = []

for column in numeric_columns:
    original = roads_work[column]
    values = pd.to_numeric(original, errors="coerce")

    numeric_checks.append({
        "field": column,
        "missing": original.isna().sum(),
        "non_numeric": (original.notna() & values.isna()).sum(),
        "zero": (values == 0).sum(),
        "negative": (values < 0).sum(),
        "min": values.min(),
        "max": values.max()
    })

display(pd.DataFrame(numeric_checks))

,field,missing,non_numeric,zero,negative,min,max
0,Length,0,0,0,0,6.000000e-07,2.751000
1,Shape__Length,35,0,0,0,2.831960e-03,4427.314654
2,Num_Lanes,472,0,2257,0,0.000000e+00,5.000000
3,Surface_Wd,657,0,3555,0,0.000000e+00,80.000000
4,Speed_Lim,13597,0,76,0,0.000000e+00,99.000000
5,Speed,22363,0,0,0,2.500000e+01,45.000000
6,AADT,4206,0,0,0,1.000000e+01,205066.000000


In [24]:
speed_field = next(
    field for field in metadata["fields"]
    if field["name"] == "Speed_Lim"
)

print("Speed_Lim 的代碼／範圍定義：")
print(speed_field.get("domain"))

Speed_Lim 的代碼／範圍定義：
None


- Speed_Lim has no domain definition in the API metadata.
- Values of 0 and 99 require verification against the official data dictionary.
- Original values are retained pending verification.

In [25]:
geometry_checks = []

for feature in boston_features:
    attributes = feature["attributes"]
    geometry = feature.get("geometry") or {}
    paths = geometry.get("paths") or []

    geometry_checks.append({
        "OBJECTID": attributes["OBJECTID"],
        "geometry_missing_or_empty": not any(
            len(path) >= 2 for path in paths
        ),
        "shape_length_missing": (
            pd.isna(attributes.get("Shape__Length"))
        )
    })

geometry_check_df = pd.DataFrame(geometry_checks)

display(
    pd.crosstab(
        geometry_check_df["geometry_missing_or_empty"],
        geometry_check_df["shape_length_missing"],
        rownames=["座標缺失或不足"],
        colnames=["Shape__Length 缺失"]
    )
)

Shape__Length 缺失,False,True
座標缺失或不足,,
False,22592,0
True,0,35


In [26]:
geometry_flags = geometry_check_df.set_index("OBJECTID")[
    "geometry_missing_or_empty"
]

roads_work["geometry_missing_or_empty"] = (
    roads_work["OBJECTID"].map(geometry_flags)
)

print(
    "座標缺失或不足的筆數:",
    roads_work["geometry_missing_or_empty"].sum()
)

座標缺失或不足的筆數: 35


In [27]:
import pandas as pd

# 1. 建立檢查用表格：不修改原始資料
audit_df = pd.DataFrame(
    feature["attributes"] for feature in boston_features
).replace(r"^\s*$", pd.NA, regex=True)

print(f"道路紀錄總數：{len(audit_df):,}")

# 2. 全部欄位的缺失情況
missing_report = pd.DataFrame({
    "missing_count": audit_df.isna().sum(),
    "missing_percent": (audit_df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

print("\n【欄位缺失：只顯示有缺失的欄位】")
display(missing_report[missing_report["missing_count"] > 0])

# 3. ID 完整性與重複
id_rows = []

for column in ["OBJECTID", "Record_ID", "Rd_Seg_ID", "GlobalID"]:
    values = audit_df[column].dropna()
    id_rows.append({
        "field": column,
        "missing_count": audit_df[column].isna().sum(),
        "unique_non_missing": values.nunique(),
        "rows_with_repeated_id": values.duplicated(keep=False).sum()
    })

id_report = pd.DataFrame(id_rows)

print("\n【ID 檢查】")
display(id_report)

# 4. 數值檢查：零與極端值只是待確認，不直接判錯
numeric_rows = []

for column in [
    "Length", "Shape__Length", "Num_Lanes",
    "Surface_Wd", "Speed_Lim", "Speed", "AADT"
]:
    original = audit_df[column]
    values = pd.to_numeric(original, errors="coerce")

    numeric_rows.append({
        "field": column,
        "missing": original.isna().sum(),
        "non_numeric": (original.notna() & values.isna()).sum(),
        "zero": (values == 0).sum(),
        "negative": (values < 0).sum(),
        "min": values.min(),
        "max": values.max()
    })

numeric_report = pd.DataFrame(numeric_rows)

print("\n【數值檢查】")
display(numeric_report)

# 5. AADT 與測量年份
aadt = pd.to_numeric(audit_df["AADT"], errors="coerce")
years = pd.to_numeric(audit_df["AADT_Year"], errors="coerce")

aadt_report = pd.DataFrame({
    "status": [
        "AADT and year available",
        "AADT available, year missing",
        "AADT missing, year available",
        "Both missing"
    ],
    "record_count": [
        (aadt.notna() & years.notna()).sum(),
        (aadt.notna() & years.isna()).sum(),
        (aadt.isna() & years.notna()).sum(),
        (aadt.isna() & years.isna()).sum()
    ]
})

print("\n【AADT 完整性】")
display(aadt_report)

print("\n【AADT 年份分布】")
display(
    years.value_counts().sort_index()
    .rename_axis("AADT_Year")
    .reset_index(name="record_count")
)
print("年份晚於 2022 的筆數：", (years > 2022).sum())

# 6. 設施類型分布
facility_field = next(
    field for field in metadata["fields"]
    if field["name"] == "Facility"
)
codes = (facility_field.get("domain") or {}).get("codedValues", [])
name_map = {item["code"]: item["name"] for item in codes}

facility_report = (
    audit_df.groupby("Facility", dropna=False)
    .size().reset_index(name="record_count")
)
facility_report["facility_name"] = (
    facility_report["Facility"].map(name_map)
)
facility_report["percent"] = (
    facility_report["record_count"] / len(audit_df) * 100
).round(2)

print("\n【設施類型】")
display(facility_report.sort_values("record_count", ascending=False))

# 7. 基本座標檢查：不等同完整幾何有效性檢查
geometry_rows = []

for feature in boston_features:
    attributes = feature["attributes"]
    geometry = feature.get("geometry") or {}
    paths = geometry.get("paths") or []

    geometry_rows.append({
        "OBJECTID": attributes["OBJECTID"],
        "geometry_missing_or_empty": not any(
            len(path) >= 2 for path in paths
        ),
        "shape_length_missing": pd.isna(
            attributes.get("Shape__Length")
        )
    })

geometry_report = pd.DataFrame(geometry_rows)

print("\n【基本座標與長度完整性】")
display(
    pd.crosstab(
        geometry_report["geometry_missing_or_empty"],
        geometry_report["shape_length_missing"],
        rownames=["座標缺失或不足"],
        colnames=["Shape__Length 缺失"]
    )
)

print("\n檢查完成；未刪除紀錄、未補值、未修改原始檔。")

道路紀錄總數：22,627

【欄位缺失：只顯示有缺失的欄位】


,missing_count,missing_percent
Structure,22627,100.00
Path_Width,22627,100.00
Cr_Slope,22627,100.00
Pd_Sf_Type,22611,99.93
Pd_Fc_Type,22598,99.87
Trk_Netwrk,22400,99.00
Toll_Road,22376,98.89
Speed,22363,98.83
MHS,22146,97.87
StationID,22004,97.25



【ID 檢查】


,field,missing_count,unique_non_missing,rows_with_repeated_id
0,OBJECTID,0,22627,0
1,Record_ID,37,22590,0
2,Rd_Seg_ID,13699,6660,3925
3,GlobalID,0,22627,0



【數值檢查】


...

年份晚於 2022 的筆數： 0

【設施類型】


,Facility,record_count,facility_name,percent
0,1.0,18398,Mainline roadway,81.31
10,NaN,3288,NaN,14.53
4,7.0,390,Simple Ramp/ Channelized Turning Lane,1.72
9,12.0,246,Bicycle/ Pedestrian,1.09
1,3.0,96,Tunnel,0.42
3,5.0,74,Rotary,0.33
8,11.0,73,Simple Ramp - Tunnel,0.32
5,8.0,25,Ramp - NB/EB,0.11
2,4.0,21,Doubledeck,0.09
7,10.0,11,Collector - Distributor,0.05



【基本座標與長度完整性】


Shape__Length 缺失,False,True
座標缺失或不足,,
False,22592,0
True,0,35



檢查完成；未刪除紀錄、未補值、未修改原始檔。


In [1]:
import json
import pandas as pd
from pathlib import Path

# Locate the project root.
cwd = Path.cwd()
project_root = next(
    (p for p in [cwd, *cwd.parents] if (p / ".git").exists()),
    None
)
if project_root is None:
    raise RuntimeError("Project root not found. Open the project folder in VS Code.")

# Load the raw snapshot saved earlier.
raw_path = (
    project_root / "data" / "raw"
    / "boston_roads_2022_raw_20261016T170046534733Z.json"
)

with raw_path.open("r", encoding="utf-8") as file:
    raw_data = json.load(file)

boston_features = raw_data["features"]
metadata = raw_data["layer_metadata"]

boston_df = pd.DataFrame(
    feature["attributes"] for feature in boston_features
)

assert len(boston_df) == raw_data["record_count"]
print("Loaded records:", len(boston_df))
print("Source file:", raw_path.name)

Loaded records: 22627
Source file: boston_roads_2022_raw_20261016T170046534733Z.json


In [2]:
from datetime import datetime, timezone

# Create a preliminary working copy.
roads_preliminary = boston_df.copy()
roads_preliminary["segment_id"] = roads_preliminary["GlobalID"]

# Preserve original names and standardize blank names.
roads_preliminary["street_name_clean"] = (
    roads_preliminary["St_Name"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

# Flag missing values without imputing them.
roads_preliminary["street_name_missing"] = (
    roads_preliminary["street_name_clean"].isna()
)

for source, flag in [
    ("AADT", "aadt_missing"),
    ("AADT_Year", "aadt_year_missing"),
    ("Facility", "facility_missing")
]:
    roads_preliminary[flag] = roads_preliminary[source].isna()

# Check basic geometry completeness, not full geometric validity.
geometry_flags = {}
for feature in boston_features:
    geometry = feature.get("geometry") or {}
    paths = geometry.get("paths") or []
    geometry_flags[feature["attributes"]["OBJECTID"]] = not any(
        len(path) >= 2 for path in paths
    )

roads_preliminary["geometry_missing_or_empty"] = (
    roads_preliminary["OBJECTID"].map(geometry_flags)
)

# Validate record identity and preservation.
assert len(roads_preliminary) == raw_data["record_count"]
assert roads_preliminary["segment_id"].notna().all()
assert roads_preliminary["segment_id"].is_unique
assert roads_preliminary["geometry_missing_or_empty"].notna().all()

# Create a new output folder for this run.
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir = (
    project_root / "data" / "processed" / f"boston_preliminary_{stamp}"
)
output_dir.mkdir(parents=True, exist_ok=False)

# Save attributes for tabular analysis.
roads_preliminary.to_csv(
    output_dir / "roads_preliminary_attributes.csv",
    index=False,
    encoding="utf-8-sig"
)

# Preserve the original geometry payload and metadata.
# This is Esri JSON, not GeoJSON.
with (output_dir / "source_features_esri.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(raw_data, file, ensure_ascii=False)

# Save a concise quality-flag summary.
flag_columns = [
    "street_name_missing", "aadt_missing",
    "aadt_year_missing", "facility_missing",
    "geometry_missing_or_empty"
]

flag_summary = pd.DataFrame({
    "flagged_count": roads_preliminary[flag_columns].sum(),
    "flagged_percent": (
        roads_preliminary[flag_columns].mean() * 100
    ).round(2)
})
flag_summary.to_csv(output_dir / "quality_flags.csv", index_label="flag")

print("Saved to:", output_dir)
print("Records preserved:", len(roads_preliminary))
display(flag_summary)
print("Preliminary only: no records removed or values imputed.")

Saved to: c:\Users\wendy\Desktop\Boston University\classes\CS506\Final_Project\cs506-final-project\data\processed\boston_preliminary_20261017T013359153208Z
Records preserved: 22627


,flagged_count,flagged_percent
street_name_missing,2800,12.37
aadt_missing,4206,18.59
aadt_year_missing,14104,62.33
facility_missing,3288,14.53
geometry_missing_or_empty,35,0.15


Preliminary only: no records removed or values imputed.
